# EdgeGuard · Drive veri ön-hazırlığı

Bu notebook veri indirme yetkisi vermez ve lisans kabulünü otomatikleştirmez. Resmî paketleri Drive'a yerleştirdikten sonra klasör düzenini denetler ve her veri setini tek, SHA-256 bağlı `.tar` dosyasına dönüştürür. Eğitim notebook'u Drive'daki binlerce küçük dosyayı okumak yerine bu paketleri `/content` alanına taşır.

Çekirdek eğitim için yalnız **Cityscapes Fine + BDD100K 10K Semantic + IDD20K Part I/II** gerekir. ACDC ve kapalı external setler model dondurulmadan indirilmez.

In [ ]:
import os
import sys
from pathlib import Path

LOCAL_TEST_MODE = os.environ.get("EDGEGUARD_NOTEBOOK_LOCAL_TEST") == "1"
if LOCAL_TEST_MODE:
    PROJECT_ROOT = Path(os.environ.get("EDGEGUARD_PROJECT_ROOT", Path.cwd())).resolve()
    DRIVE_ROOT = Path(os.environ["EDGEGUARD_TEST_DRIVE_ROOT"]).resolve()
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/edgeguard-road")
    DRIVE_ROOT = Path("/content/drive/MyDrive")

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "rescue/semantic-first"
EXPECTED_PROJECT_COMMIT = "6745106f0cb419dcc15b51cc3856eb87042086a6"
SCIENTIFIC_SOURCE_DATASETS = ["cityscapes", "idd20k"]
PROVISIONAL_ENGINEERING_DATASETS = ["bdd100k"]
OPTIONAL_FINAL_DATASETS = []  # Model/protokol freeze sonrası ör. ["acdc"]
DATASETS_TO_BUNDLE = ["cityscapes", "bdd100k", "idd20k"] + OPTIONAL_FINAL_DATASETS
VERIFY_ARCHIVE_HASHES = True  # Resmî arşivleri bir kez SHA-256/MD5 ile kaydeder.
RUN_ARCHIVE_PREPARATION = False  # Arşivler Drive'a yüklendikten sonra bir kez True yapın.
BDD_SOURCE_PROFILE = "kaggle_mirror"  # Drive'daki bdd100k.zip; yalnız audit/smoke kanıtıdır.
CREATE_BUNDLES = True  # Hazırlanan yerel kökten doğrudan tek Drive tar üretir.
REPLACE_BUNDLES = False  # Yalnız kaynak klasörü bilinçli değiştiyse True yapın.
REUSE_VERIFIED_LEGACY = True  # Mevcut hash-bağlı Cityscapes bundle'ını yeniden kullanır.
REPAIR_STALE_EPHEMERAL_PREPARATION = True  # Yalnız iki sabit /content çalışma kökünü temizler.
DOWNLOAD_LATEST_FAILURE_REPORT = False  # Hata sonrası son hücreyi bununla yeniden çalıştırın.


def persist_bootstrap_failure(notebook, stage, error):
    import json
    import re
    import traceback
    import uuid
    from datetime import datetime, timezone
    from zipfile import ZIP_DEFLATED, ZipFile

    failed_at = datetime.now(timezone.utc)
    failure_id = f"{failed_at.strftime('%Y%m%dT%H%M%S.%fZ')}-{stage}-{uuid.uuid4().hex[:8]}"
    root = DRIVE_ROOT / "EdgeGuard/failures/bootstrap" / failure_id
    root.mkdir(parents=True, exist_ok=False)
    rendered = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    rendered = re.sub(r"(?i)(token|password|secret|api[_-]?key)=\S+", r"\1=<redacted>", rendered)
    payload = {
        "record_type": "edgeguard_colab_bootstrap_failure",
        "failure_id": failure_id,
        "failed_at": failed_at.isoformat(),
        "notebook": notebook,
        "stage": stage,
        "error_type": type(error).__name__,
        "traceback": rendered,
    }
    report = root / "failure.json"
    report.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    package = root / "failure-report.zip"
    with ZipFile(package, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(report, arcname="failure.json")
    print("EDGEGUARD BOOTSTRAP FAILURE:", package)
    return package

In [ ]:
import subprocess


def run_bootstrap_command(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    output = "\n".join(
        value.strip() for value in (completed.stdout, completed.stderr) if value.strip()
    )
    if output:
        print(output)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Bootstrap command failed with exit code {completed.returncode}: {command}\n"
            + output[-8000:]
        )
    return completed


try:
    if LOCAL_TEST_MODE:
        print("LOCAL_TEST_MODE: Drive mount, clone ve paket kurulumu atlandı.")
    elif not (PROJECT_ROOT / ".git").is_dir():
        run_bootstrap_command(["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)])
    else:
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH])
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH])
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"])
except BaseException as error:
    persist_bootstrap_failure("EdgeGuard_Data_Preflight_Colab.ipynb", "git-clone-or-update", error)
    raise
if not LOCAL_TEST_MODE:
    if EXPECTED_PROJECT_COMMIT:
        run_bootstrap_command(["git", "-C", str(PROJECT_ROOT), "checkout", EXPECTED_PROJECT_COMMIT])
PROJECT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from edgeguard.rescue.colab_failures import (  # noqa: E402
    ColabFailureReporter,
    run_logged_command,
)

FAILURE_REPORTER = ColabFailureReporter(
    DRIVE_ROOT / "EdgeGuard/failures/data-preflight" / PROJECT_COMMIT,
    notebook="EdgeGuard_Data_Preflight_Colab.ipynb",
    project_commit=PROJECT_COMMIT,
    context={"branch": BRANCH, "local_test_mode": LOCAL_TEST_MODE},
)
COMMAND_LOG_ROOT = (
    Path(os.environ.get("EDGEGUARD_TEST_CONTENT_ROOT", "/content")) / "edgeguard-command-logs"
)
FAILURE_REPORTER.add_diagnostic_root("manifests", DRIVE_ROOT / "EdgeGuard/manifests")
FAILURE_REPORTER.add_diagnostic_root("command-logs", COMMAND_LOG_ROOT)
FAILURE_REPORTER.install_ipython_hook()


def run_colab_command(command, *, check=True):
    command_env = os.environ.copy()
    project_src = str(PROJECT_ROOT / "src")
    command_env["PYTHONPATH"] = project_src + os.pathsep + command_env.get("PYTHONPATH", "")
    return run_logged_command(
        command,
        log_root=COMMAND_LOG_ROOT,
        stage=FAILURE_REPORTER.stage,
        check=check,
        cwd=PROJECT_ROOT,
        env=command_env,
    )


if not LOCAL_TEST_MODE:
    FAILURE_REPORTER.set_stage("project-install")
    run_colab_command([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)])

In [ ]:
# Drive klasörlerini oluştur, erişim talimatlarını ve eksikleri tek raporda göster.
import json

from edgeguard.rescue.colab_data import load_colab_data_access

FAILURE_REPORTER.set_stage("drive-inventory-and-hashing")
PREFLIGHT_REPORT = DRIVE_ROOT / "EdgeGuard/manifests/colab-data-inventory.json"
inventory_plan = load_colab_data_access(PROJECT_ROOT / "configs/dataset/colab_data_access_v1.yaml")
inventory_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
    "--drive-root",
    str(DRIVE_ROOT),
    "--output",
    str(PREFLIGHT_REPORT),
    "inventory",
]
if VERIFY_ARCHIVE_HASHES:
    inventory_command.append("--hash-archives")
run_colab_command(inventory_command)
inventory = json.loads(PREFLIGHT_REPORT.read_text())
for row in inventory["datasets"]:
    print("\n", row["dataset_id"], "=>", row["state"], "|", row["activation_phase"])
    print("resmî kaynak:", row["official_url"])
    print("işlem:", row["instructions"])
    if row["missing_required_paths"]:
        print("eksik hazır yollar:", row["missing_required_paths"])
    for package in row["packages"]:
        print(
            "paket:",
            package["filename"],
            "Drive'da:",
            package["present"],
            "konum:",
            package["location_profile"],
            "hash:",
            package["hash_status"],
        )
        if package.get("hash_error"):
            print("hash okuma uyarısı:", package["hash_error"])
    for package in row.get("engineering_packages", []):
        print(
            "mühendislik paketi:",
            package["filename"],
            "Drive'da:",
            package["present"],
            "profil:",
            package["source_profile"],
            "bilimsel:",
            package["scientific_eligible"],
            "hash:",
            package["hash_status"],
        )
        if package.get("hash_error"):
            print("hash okuma uyarısı:", package["hash_error"])
    if row.get("legacy_compatibility"):
        print("legacy uyumluluk:", row["legacy_compatibility"])

## Arşivden güvenli hazırlama

Notebook hem yeni `MyDrive/EdgeGuard/archives/<dataset_id>/` düzenini hem de mevcut `MyDrive/EdgeGuard/private_inputs/` klasörünü salt-okunur girdi olarak tanır. Giriş bilgisi, cookie veya geçici indirme URL'sini notebook'a yazmayın. Datasetleri elle açmayın. `RUN_ARCHIVE_PREPARATION=True` olduğunda notebook arşivleri sırayla `/content` alanına kopyalar, hash doğrular, güvenli biçimde hazırlar ve doğrudan tek dosyalı Drive bundle üretir.

Mevcut `private_inputs/bdd100k.zip` Kaggle kaynağıdır. Bundle smoke/plumbing ve veri kataloğu için hazırlanır fakat bilimsel manifest olamaz. Ana bilimsel kaynaklar bu nedenle Cityscapes + IDD20K olarak ayarlanmıştır; resmî iki BDD paketi daha sonra gelirse BDD yeniden ana karşılaştırmaya alınabilir.

IDD polygon JSON etiketleri pinned AutoNUE source-ID sözleşmesiyle maskeye çevrilir; Part II JPG görüntüleri korunur. Native polygon ve source-ID maskeler ayrı kalır.

In [ ]:
# Arşivleri dataset bazında hazırla; büyük ağaçları Drive'a küçük dosyalar hâlinde yazma.
import contextlib
import shutil
import threading
import time

from edgeguard.rescue.colab_data import copy_archive_to_local, preparation_disk_budget
from edgeguard.serialization import canonical_json, sha256_file

FAILURE_REPORTER.set_stage("dataset-preparation-and-bundling")
CONTENT_ROOT = Path(os.environ.get("EDGEGUARD_TEST_CONTENT_ROOT", "/content"))
PREPARE_ROOT = CONTENT_ROOT / "edgeguard-prepare"
CACHE_ROOT = CONTENT_ROOT / "edgeguard-archive-cache"
archive_root = DRIVE_ROOT / "EdgeGuard/archives"


def _tree_progress(roots):
    files = 0
    byte_size = 0
    for root in roots:
        if not root.exists():
            continue
        for directory, _subdirectories, filenames in os.walk(root):
            for filename in filenames:
                try:
                    byte_size += (Path(directory) / filename).stat().st_size
                    files += 1
                except OSError:
                    continue
    return files, byte_size


@contextlib.contextmanager
def live_preparation_progress(dataset, phase, roots, interval_seconds=60):
    # Print bounded liveness evidence while a quiet archive subprocess runs.
    stopped = threading.Event()
    started = time.monotonic()

    def report():
        while not stopped.wait(interval_seconds):
            files, byte_size = _tree_progress(roots)
            free = shutil.disk_usage(CONTENT_ROOT).free
            print(
                f"EDGEGUARD PROGRESS dataset={dataset} phase={phase['value']} "
                f"elapsed_min={(time.monotonic() - started) / 60:.1f} "
                f"files={files} bytes={byte_size} free_bytes={free}",
                flush=True,
            )

    print(
        f"{dataset}: otomatik canlı durum satırı her {interval_seconds} saniyede yazılacak. "
        "Dosya/byte sayısı artmıyorsa aynı komutu yeniden başlatmayın.",
        flush=True,
    )
    print(
        "Gerekirse Colab terminalinde kontrol edin: "
        "ps -eo pid,etime,%cpu,%mem,stat,cmd | grep '[p]repare_dataset.py'; "
        f"du -sh {PREPARE_ROOT} {CACHE_ROOT}; df -h {CONTENT_ROOT}",
        flush=True,
    )
    worker = threading.Thread(target=report, name=f"edgeguard-progress-{dataset}", daemon=True)
    worker.start()
    try:
        yield
    finally:
        stopped.set()
        worker.join(timeout=2)
        files, byte_size = _tree_progress(roots)
        print(
            f"EDGEGUARD PROGRESS dataset={dataset} phase={phase['value']} "
            f"elapsed_min={(time.monotonic() - started) / 60:.1f} "
            f"files={files} bytes={byte_size} final=True",
            flush=True,
        )


def _cache_receipt_path(local_archive):
    return local_archive.with_name(local_archive.name + ".copy-receipt.json")


def reuse_or_copy_archive(source, local, source_record):
    # Reuse only a same-source, same-size, hash-verified ephemeral archive copy.
    receipt_path = _cache_receipt_path(local)
    expected_sha256 = source_record.get("sha256") or source_record.get("published_sha256")
    reusable = False
    if local.is_file() and not local.is_symlink():
        try:
            same_size = local.stat().st_size == source.stat().st_size
            if receipt_path.is_file():
                receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
                reusable = (
                    receipt.get("source") == str(source.resolve())
                    and int(receipt.get("byte_size", -1)) == source.stat().st_size
                    and same_size
                    and receipt.get("expected_sha256") == expected_sha256
                    and receipt.get("copied_sha256") == expected_sha256
                )
            else:
                # A cache produced by the previous notebook has no sidecar. A pinned
                # or freshly inventoried digest is sufficient to adopt it safely.
                reusable = same_size and expected_sha256 is not None
            if reusable and expected_sha256 and not receipt_path.is_file():
                print("Yerel cache SHA-256 doğrulanıyor:", local.name, flush=True)
                reusable = sha256_file(local) == expected_sha256
        except (OSError, TypeError, ValueError, json.JSONDecodeError):
            reusable = False
    if reusable:
        print("Doğrulanmış /content arşiv cache'i yeniden kullanılıyor:", local.name)
        return {
            "source": str(source),
            "destination": str(local),
            "byte_size": local.stat().st_size,
            "status": "reused",
        }
    local.unlink(missing_ok=True)
    receipt_path.unlink(missing_ok=True)
    print("Drive arşivi /content alanına kopyalanıyor:", source.name)
    copy_receipt = copy_archive_to_local(source, local, attempts=3)
    cache_receipt = {
        "source": str(source.resolve()),
        "byte_size": source.stat().st_size,
        "expected_sha256": expected_sha256,
        "copied_sha256": copy_receipt["sha256"],
    }
    receipt_path.write_text(canonical_json(cache_receipt) + "\n", encoding="utf-8")
    return {**copy_receipt, "status": "copied"}


if RUN_ARCHIVE_PREPARATION:
    if PREPARE_ROOT.exists():
        if not REPAIR_STALE_EPHEMERAL_PREPARATION:
            raise RuntimeError("Stale preparation root found; inspect before retrying")
        if PREPARE_ROOT.is_symlink() or PREPARE_ROOT.name != "edgeguard-prepare":
            raise RuntimeError(f"Refusing unsafe preparation cleanup: {PREPARE_ROOT}")
        shutil.rmtree(PREPARE_ROOT)
    if CACHE_ROOT.is_symlink() or CACHE_ROOT.name != "edgeguard-archive-cache":
        raise RuntimeError(f"Refusing unsafe archive cache: {CACHE_ROOT}")
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    for dataset in DATASETS_TO_BUNDLE:
        row = next(item for item in inventory["datasets"] if item["dataset_id"] == dataset)
        legacy = row.get("legacy_compatibility") or {}
        if REUSE_VERIFIED_LEGACY and legacy.get("usable_for_training_staging"):
            print(dataset, "mevcut doğrulanmış legacy bundle ile yeniden kullanılacak.")
            continue
        if (
            dataset == "bdd100k"
            and BDD_SOURCE_PROFILE == "kaggle_mirror"
            and row.get("ineligible_smoke_bundle_usable")
        ):
            print(dataset, "mevcut provisional mirror bundle ile yeniden kullanılacak.")
            continue
        if row.get("canonical_bundle_usable") and (
            dataset != "bdd100k" or BDD_SOURCE_PROFILE == "official"
        ):
            print(dataset, "mevcut canonical bundle ile yeniden kullanılacak.")
            continue
        if dataset == "bdd100k" and BDD_SOURCE_PROFILE == "kaggle_mirror":
            engineering = [
                item
                for item in row["engineering_packages"]
                if item["source_profile"] == "kaggle_mirror"
            ]
            source_records = engineering
        else:
            source_records = row["packages"]
        sources = [Path(item["path"]) for item in source_records]
        missing = [str(path) for path in sources if not path.is_file()]
        if missing:
            raise FileNotFoundError("Missing archives: " + ", ".join(missing))
        budget = preparation_disk_budget(inventory_plan, tuple(sources), CONTENT_ROOT)
        print(dataset, "hazırlık disk kapısı:", budget)
        dataset_cache = CACHE_ROOT / dataset
        dataset_cache.mkdir(parents=True, exist_ok=True)
        local_archives = []
        for source, source_record in zip(sources, source_records, strict=True):
            local = dataset_cache / source.name
            copy_receipt = reuse_or_copy_archive(source, local, source_record)
            print("Arşiv kopyası tamamlandı:", copy_receipt)
            local_archives.append(local)
        prepared = PREPARE_ROOT / dataset
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_dataset.py"),
            "--dataset",
            dataset,
            "--destination",
            str(prepared),
            "--ontology",
            str(PROJECT_ROOT / "configs/dataset/semantic_ontology_v2.yaml"),
        ]
        for archive in local_archives:
            command.extend(["--archive", str(archive)])
        if dataset == "bdd100k":
            command.extend(["--source-profile", BDD_SOURCE_PROFILE])
        phase = {"value": "archive-verify-extract-map"}
        progress_roots = (dataset_cache, PREPARE_ROOT / f".{dataset}.incoming", prepared)
        with live_preparation_progress(dataset, phase, progress_roots):
            run_colab_command(command)
            if CREATE_BUNDLES:
                phase["value"] = "bundle-write-and-hash"
                bundle = [
                    sys.executable,
                    str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
                    "--drive-root",
                    str(DRIVE_ROOT),
                    "bundle",
                    "--dataset",
                    dataset,
                    "--source-root",
                    str(prepared),
                ]
                if REPLACE_BUNDLES:
                    bundle.append("--replace")
                run_colab_command(bundle)
        shutil.rmtree(dataset_cache)
        shutil.rmtree(prepared)
    if CACHE_ROOT.is_dir():
        remaining_cache = sorted(path.name for path in CACHE_ROOT.iterdir())
        if remaining_cache:
            print("Yeniden deneme için korunan yerel arşiv cache'leri:", remaining_cache)
        else:
            CACHE_ROOT.rmdir()
    if PREPARE_ROOT.is_dir():
        PREPARE_ROOT.rmdir()
    run_colab_command(inventory_command)
    inventory = json.loads(PREFLIGHT_REPORT.read_text())
else:
    print("RUN_ARCHIVE_PREPARATION=False: arşiv yükleme ve inventory incelemesi bekleniyor.")

In [ ]:
# Eğitim notebook'una geçiş kapısı: bilimsel ve provisional durumlar ayrı raporlanır.
FAILURE_REPORTER.set_stage("preflight-readiness-gate")
bundle_root = DRIVE_ROOT / "EdgeGuard/bundles"
missing = []
for dataset in SCIENTIFIC_SOURCE_DATASETS:
    row = next(item for item in inventory["datasets"] if item["dataset_id"] == dataset)
    legacy = row.get("legacy_compatibility") or {}
    if row.get("canonical_bundle_usable") or legacy.get("usable_for_training_staging"):
        continue
    for suffix in (".prepared.tar", ".prepared.tar.receipt.json"):
        candidate = bundle_root / f"{dataset}{suffix}"
        if not candidate.is_file():
            missing.append(str(candidate))
if missing:
    print("Bilimsel eğitim öncesi eksikler:\n- " + "\n- ".join(missing))
else:
    print("BİLİMSEL VERİ KAPISI GEÇTİ — Cityscapes + IDD20K hazır.")
bdd_row = next(item for item in inventory["datasets"] if item["dataset_id"] == "bdd100k")
print("BDD provisional mirror bundle:", bdd_row.get("ineligible_smoke_bundle_usable", False))
print("BDD resmî bilimsel bundle:", bdd_row.get("canonical_bundle_usable", False))
print("Hata raporu kökü:", FAILURE_REPORTER.output_root)
if DOWNLOAD_LATEST_FAILURE_REPORT:
    latest_failure = FAILURE_REPORTER.latest_package()
    if latest_failure is None:
        raise RuntimeError("İndirilecek hata paketi bulunamadı")
    if not LOCAL_TEST_MODE:
        from google.colab import files

        files.download(str(latest_failure))
    print("Hata paketi:", latest_failure)